### Loading & Merge DataFrame

In [1]:
import pandas as pd
headline_df = pd.read_csv("/content/drive/MyDrive/financial_data/finbert_emb_v2.csv", index_col = 0).rename(columns={'stock_name': 'stock'})
ohlcv_df = pd.read_csv("/content/drive/MyDrive/financial_data/ohlcv.csv", index_col = 0)
is_easy_df = pd.read_csv("/content/drive/MyDrive/financial_data/is_easy_1_5_20_v2.csv", index_col = 0)[['stock', 'date', 'easy_indicator']]

# --- Data Preparation ---
# Create a mapping of Date to general file_path
general_paths = headline_df[headline_df['stock'] == 'general'][['Date', 'file_path']].set_index('Date')['file_path'].reset_index().rename(columns={'Date': 'date', 'file_path': 'file_path_general'})

# Only get the specific filepath
headline_df = headline_df[['Date', 'stock', 'file_path']]

headline_df['Date'] = pd.to_datetime(headline_df['Date'])

# Convert Date column to datetime if not already
headline_df['Date'] = pd.to_datetime(headline_df['Date'])

# Define date range
start_date = '2010-01-06'
end_date = '2020-07-19'

# Filter the DataFrame
headline_df = headline_df[(headline_df['Date'] >= start_date) & (headline_df['Date'] <= end_date)]


# 1. Convert date columns to datetime objects for proper merging and filtering
headline_df['Date'] = pd.to_datetime(headline_df['Date'])
ohlcv_df['date'] = pd.to_datetime(ohlcv_df['date'])
is_easy_df['date'] = pd.to_datetime(is_easy_df['date'])
general_paths['date'] = pd.to_datetime(general_paths['date'])
# 2. Rename headline date column to match the others
headline_df = headline_df.rename(columns={'Date': 'date'})

# 3. Optional: Drop unnecessary columns from headline_df
headline_df = headline_df[['date', 'stock', 'file_path']]

# --- Merging Process ---

# Step 1: Merge ohlcv_df and is_easy_df on 'date' and 'stock'
# Using 'inner' merge keeps only rows where a stock exists on a specific date in BOTH dataframes
merged_ohlcv_easy = pd.merge(ohlcv_df, is_easy_df, on=['date', 'stock'], how='inner')

# Step 2: Merge the result with headline_df on 'date'
# Using 'inner' merge ensures that we only keep data for dates present in headline_df.
# This automatically restricts the final dataframe to the desired time range.
merged_df = pd.merge(merged_ohlcv_easy, headline_df, on=['date', 'stock'], how='left')

# Get the general paths
merged_df = pd.merge(merged_df, general_paths, on=['date'], how='left')

# 3. Determine the minimum and maximum dates from headline_df
min_date = headline_df['date'].min()
max_date = headline_df['date'].max()
final_merged_df = merged_df[(merged_df['date'] >= min_date) & (merged_df['date'] <= max_date)]

# 4. Confirmed the stocks are within the list
stocks = [
    "PLTR", "TSLA", "KKR", "AMD", "NVDA", "BX", "BA", "AMAT", "LRCX", "C",
    "DIS", "BKNG", "ISRG", "GS", "UBER", "MS", "BAC", "ADBE", "CRM", "BLK",
    "NFLX", "KLAC", "QCOM", "INTU", "AXP", "ACN", "GE", "SPGI", "AAPL", "META",
    "COP", "MU", "WFC", "CRWD", "AMZN", "PANW", "PLD", "JPM", "CAT", "LOW",
    "MA", "CVX", "ANET", "INTC", "UNP", "HD", "ORCL", "HON", "ETN", "ADI"
]

final_merged_df = final_merged_df[final_merged_df['stock'].isin(stocks)]

### Sparse Auto-Encoder Evaluation

In [2]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import pandas as pd
from tqdm import tqdm
from collections import defaultdict
from transformers import PatchTSMixerConfig, PatchTSMixerForTimeSeriesClassification
from torch import nn, optim
from sklearn.metrics import accuracy_score, f1_score

class StockClassificationDataset(Dataset):
    def __init__(self, df, sequence_length=252, forecast_horizon=1,
                 feature_cols=['open', 'high', 'low', 'close', 'volume'],
                 sentiment_dim=768, output_dir='/content/drive/MyDrive/financial_data/embedding_cache', dataset_type='train'):
        """
        Initialize the dataset with preloaded embeddings, saving/loading caches to/from output_dir.

        Parameters:
        - df: DataFrame with columns: date, stock, open, high, low, close, volume, easy_indicator, file_path, file_path_general
        - sequence_length: Number of days in each sequence (default: 252)
        - forecast_horizon: Number of days ahead to predict (default: 1)
        - feature_cols: List of OHLCV feature columns
        - sentiment_dim: Dimensionality of sentiment embeddings (default: 768)
        - output_dir: Directory to save/load embedding cache files (default: './embedding_cache')
        - dataset_type: 'train' or 'test' to distinguish cache files (default: 'train')
        """
        assert sequence_length >= forecast_horizon, "Sequence length must be at least as long as forecast_horizon."

        self.sequence_length = sequence_length
        self.forecast_horizon = forecast_horizon
        self.feature_cols = feature_cols
        self.sentiment_dim = sentiment_dim
        self.output_dir = output_dir
        self.dataset_type = dataset_type

        # Create output directory if it doesn't exist
        os.makedirs(output_dir, exist_ok=True)

        # Preprocess DataFrame: remove duplicates and sort
        df['date'] = pd.to_datetime(df['date'])
        self.df = df.drop_duplicates(subset=['stock', 'date'], keep='first').sort_values(['stock', 'date']).reset_index(drop=True)

        # Store unique dates and stock data
        self.unique_dates = sorted(self.df['date'].unique())
        self.stock_data = {stock: group.reset_index(drop=True) for stock, group in self.df.groupby('stock')}

        # Define cache file paths with dataset_type suffix
        specific_cache_path = os.path.join(output_dir, f'specific_embeddings_{dataset_type}.npz')
        general_cache_path = os.path.join(output_dir, f'general_embeddings_{dataset_type}.npz')

        # Try to load embedding caches
        self.specific_embedding_cache = {}
        self.general_embedding_cache = {}
        caches_loaded = False

        if os.path.exists(specific_cache_path) and os.path.exists(general_cache_path):
            try:
                # Load specific embeddings
                specific_npz = np.load(specific_cache_path, allow_pickle=True)
                self.specific_embedding_cache = {key: specific_npz[key] for key in specific_npz.files}

                # Load general embeddings
                general_npz = np.load(general_cache_path, allow_pickle=True)
                self.general_embedding_cache = {key: general_npz[key] for key in general_npz.files}

                print(f"Loaded embedding caches from {specific_cache_path} and {general_cache_path}")
                caches_loaded = True
            except Exception as e:
                print(f"Error loading caches: {e}. Rebuilding caches...")

        if not caches_loaded:
            # Preload stock-specific embeddings (file_path)
            unique_specific_paths = self.df['file_path'].dropna().unique()
            for fp in tqdm(unique_specific_paths, desc=f"Preloading stock-specific embeddings ({dataset_type})"):
                if os.path.exists(fp):
                    emb = torch.load(fp)
                    if isinstance(emb, torch.Tensor):
                        emb = emb.cpu().numpy()
                    self.specific_embedding_cache[fp] = emb

            # Preload general embeddings (file_path_general)
            unique_general_paths = self.df['file_path_general'].dropna().unique()
            for fp in tqdm(unique_general_paths, desc=f"Preloading general embeddings ({dataset_type})"):
                if os.path.exists(fp):
                    emb = torch.load(fp)
                    if isinstance(emb, torch.Tensor):
                        emb = emb.cpu().numpy()
                    self.general_embedding_cache[fp] = emb

            # Save caches to .npz files
            try:
                np.savez_compressed(specific_cache_path, **self.specific_embedding_cache)
                np.savez_compressed(general_cache_path, **self.general_embedding_cache)
                print(f"Saved embedding caches to {specific_cache_path} and {general_cache_path}")
            except Exception as e:
                print(f"Error saving caches: {e}")

        # Build valid sequence indices
        self.valid_sequences = []
        for forecast_date in tqdm(self.unique_dates, desc="Processing Dates"):
            forecast_idx = self.unique_dates.index(forecast_date)
            if forecast_idx < sequence_length:
                continue  # Skip dates without enough prior data

            start_date = self.unique_dates[forecast_idx - sequence_length]
            for stock, stock_df in self.stock_data.items():
                seq_df = stock_df[(stock_df['date'] >= start_date) & (stock_df['date'] < forecast_date)]
                forecast_row = stock_df[stock_df['date'] == forecast_date]

                if (len(seq_df) == self.sequence_length and
                    not forecast_row.empty and
                    forecast_row.index[0] + self.forecast_horizon - 1 < len(stock_df)):
                    stock_idx = forecast_row.index[0]
                    self.valid_sequences.append({
                        'forecast_date': forecast_date,
                        'stock': stock,
                        'forecast_idx': stock_idx
                    })

        # Sort sequences by forecast date for batching
        self.valid_sequences.sort(key=lambda x: x['forecast_date'])

    def __len__(self):
        return len(self.valid_sequences)

    def __getitem__(self, idx):
        """Load data on-demand for a specific sequence."""
        seq_info = self.valid_sequences[idx]
        forecast_date = seq_info['forecast_date']
        stock = seq_info['stock']
        forecast_idx = seq_info['forecast_idx']
        stock_df = self.stock_data[stock]

        # Extract sequence
        start_idx = forecast_idx - self.sequence_length
        seq_df = stock_df.iloc[start_idx:forecast_idx]

        # Load OHLCV features
        seq_features = seq_df[self.feature_cols].values  # Shape: (sequence_length, num_features)
        scaler = StandardScaler()
        seq_features = scaler.fit_transform(seq_features)

        # Load sentiment embeddings from specific_embedding_cache
        seq_file_paths = seq_df['file_path'].values
        seq_sentiments = []
        for fp in seq_file_paths:
            emb = self.specific_embedding_cache.get(fp, np.zeros(self.sentiment_dim, dtype=np.float32))
            seq_sentiments.append(emb)
        seq_sentiments = np.stack(seq_sentiments)  # Shape: (sequence_length, sentiment_dim)

        # Load general sentiment embeddings from general_embedding_cache
        seq_file_paths_general = seq_df['file_path_general'].values
        seq_general_sentiments = []
        for fp in seq_file_paths_general:
            emb = self.general_embedding_cache.get(fp, np.zeros(self.sentiment_dim, dtype=np.float32))
            seq_general_sentiments.append(emb)
        seq_general_sentiments = np.stack(seq_general_sentiments)  # Shape: (sequence_length, sentiment_dim)

        # Compute target
        start_close = seq_df.iloc[-1]['close']
        target_close = stock_df.iloc[forecast_idx + self.forecast_horizon - 1]['close']
        pct_change = (target_close - start_close) / start_close * 100
        target = 2 if pct_change > 0.2 else (0 if pct_change < -0.2 else 1)  # Rise, Drop, Neutral
        easy_mask = stock_df.iloc[forecast_idx + self.forecast_horizon - 1]['easy_indicator']

        return {
            'features': torch.tensor(seq_features, dtype=torch.float),
            'sentiment': torch.tensor(seq_sentiments, dtype=torch.float),
            'general_sentiment': torch.tensor(seq_general_sentiments, dtype=torch.float),
            'target': torch.tensor(target, dtype=torch.long),
            'easy_mask': torch.tensor(easy_mask, dtype=torch.float),
            'forecast_date': str(forecast_date)
        }

In [3]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import pandas as pd
from tqdm import tqdm
from collections import defaultdict
from transformers import PatchTSMixerConfig, PatchTSMixerForTimeSeriesClassification
from torch import nn, optim
from sklearn.metrics import accuracy_score, f1_score, silhouette_score, classification_report
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns

# SAE Definition
class TopKSAE(nn.Module):
    def __init__(self, d_model, d_sae=2048, k=160):
        super().__init__()
        self.W_enc = nn.Parameter(torch.zeros(d_model, d_sae))
        self.W_dec = nn.Parameter(torch.zeros(d_sae, d_model))
        self.b_enc = nn.Parameter(torch.zeros(d_sae))
        self.b_dec = nn.Parameter(torch.zeros(d_model))
        self.k = k
        nn.init.xavier_uniform_(self.W_enc)
        nn.init.xavier_uniform_(self.W_dec)

    def encode(self, input_acts):
        pre_acts = input_acts @ self.W_enc + self.b_enc
        topk_values, topk_indices = torch.topk(pre_acts, self.k, dim=-1)
        acts = torch.zeros_like(pre_acts)
        acts.scatter_(-1, topk_indices, topk_values)
        return acts

    def decode(self, acts):
        return acts @ self.W_dec + self.b_dec

    def forward(self, acts):
        acts = self.encode(acts)
        recon = self.decode(acts)
        return recon

# SAE Training Function
def train_topk_sae(model, data, epochs=30, learning_rate=0.001, lambda_sparsity=0.01, batch_size=64, device='cuda'):
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.MSELoss()
    model.to(device)
    data_tensor = torch.tensor(data, dtype=torch.float32)
    dataset = torch.utils.data.TensorDataset(data_tensor)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    for epoch in range(epochs):
        total_loss = 0
        for batch in tqdm(dataloader, desc=f"SAE Epoch {epoch+1}/{epochs}"):
            inputs = batch[0].to(device)
            optimizer.zero_grad()
            recon = model(inputs)
            acts = model.encode(inputs)
            reconstruction_loss = criterion(recon, inputs)
            sparsity_loss = lambda_sparsity * torch.sum(torch.abs(acts))
            loss = reconstruction_loss + sparsity_loss
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        if epoch % 10 == 0:
            print(f"Epoch {epoch}, Loss: {total_loss / len(dataloader):.4f}")
    return model

# Updated Extract Hidden States
def extract_hidden_states(model, data_loader, device='cuda', use_sentiment=False, use_general=False, projection=None):
    model.to(device)
    model.eval()
    hidden_states_list = []
    targets = []
    predictions = []
    easy_masks = []
    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Extracting embeddings"):
            features = batch['features'].to(device)  # (batch, seq_len, 5)
            target = batch['target']
            easy_mask = batch['easy_mask']
            if use_sentiment:
                sentiment = batch['sentiment'].to(device)  # (batch, seq_len, 768)
                if projection is not None:
                    projected_sentiment = projection(sentiment)  # (batch, seq_len, 5)
                    if use_general:
                        general_sentiment = batch['general_sentiment'].to(device)  # (batch, seq_len, 768)
                        projected_general = projection(general_sentiment)  # (batch, seq_len, 5)
                        combined_input = torch.cat([features, projected_sentiment, projected_general], dim=2)  # (batch, seq_len, 15)
                    else:
                        combined_input = torch.cat([features, projected_sentiment], dim=2)  # (batch, seq_len, 10)
                    outputs = model(past_values=combined_input)
                else:
                    outputs = model(features, sentiment)
            else:
                outputs = model(past_values=features)
            hidden_states = outputs.last_hidden_state
            flattened_hidden_states = hidden_states.view(hidden_states.size(0), -1)
            pred = outputs.prediction_outputs.argmax(dim=-1)
            hidden_states_list.append(flattened_hidden_states.cpu().numpy())
            targets.append(target.numpy())
            predictions.append(pred.cpu().numpy())
            easy_masks.append(easy_mask.numpy())
    all_hidden_states = np.concatenate(hidden_states_list, axis=0)
    all_targets = np.concatenate(targets, axis=0)
    all_predictions = np.concatenate(predictions, axis=0)
    all_easy_masks = np.concatenate(easy_masks, axis=0)
    return {
        'hidden_states': all_hidden_states,
        'targets': all_targets,
        'predictions': all_predictions,
        'easy_masks': all_easy_masks
    }
# Visualization Functions
def visualize_tsne(embeddings, labels, title, filename):
    tsne = TSNE(n_components=2, random_state=42)
    tsne_embeddings = tsne.fit_transform(embeddings)
    label_to_color = {
        'Easy + Correct': 'green',
        'Easy + Incorrect': 'red',
        'Hard + Correct': 'blue',
        'Hard + Incorrect': 'orange'
    }
    colors = [label_to_color[label] for label in labels]
    plt.figure(figsize=(10, 8))
    plt.scatter(tsne_embeddings[:, 0], tsne_embeddings[:, 1], c=colors, alpha=0.5)
    plt.title(title)
    plt.xlabel("t-SNE Component 1")
    plt.ylabel("t-SNE Component 2")
    plt.legend(label_to_color.keys())
    plt.savefig(filename)
    plt.close()

def visualize_top_neurons(activation_dict, categories, top_n=10, filename_prefix=''):
    for cat in categories:
        if cat in activation_dict and len(activation_dict[cat]) > 0:
            avg_acts = activation_dict[cat].mean(axis=0)
            top_indices = np.argsort(avg_acts)[-top_n:][::-1]
            top_values = avg_acts[top_indices]
            plt.figure(figsize=(12, 6))
            plt.bar(range(top_n), top_values, tick_label=[f"Neuron {i}" for i in top_indices])
            plt.title(f"Top {top_n} Activated Neurons for {cat}")
            plt.xlabel("Neuron Index")
            plt.ylabel("Average Activation")
            plt.savefig(f"/content/drive/MyDrive/financial_data/result_v2/{filename_prefix}_{cat}_top_neurons.png")
            plt.close()

# Clustering Function
def perform_clustering(embeddings, n_clusters=4):
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    clusters = kmeans.fit_predict(embeddings)
    score = silhouette_score(embeddings, clusters)
    print(f"Silhouette Score for {n_clusters} clusters: {score:.4f}")
    return clusters

# Supervised Classification Function
def perform_classification(train_embeddings, train_labels, test_embeddings, test_labels):
    print("\nEvaluating Logistic Regression:")
    lr = LogisticRegression(max_iter=3000, random_state=42)
    lr.fit(train_embeddings, train_labels)
    lr_pred = lr.predict(test_embeddings)
    print("Logistic Regression Classification Report:")
    print(classification_report(test_labels, lr_pred))

    print("\nEvaluating Random Forest:")
    rf = RandomForestClassifier(random_state=42)
    rf.fit(train_embeddings, train_labels)
    rf_pred = rf.predict(test_embeddings)
    print("Random Forest Classification Report:")
    print(classification_report(test_labels, rf_pred))

    print("\nRecommendation: Random Forest is preferred for high-dimensional SAE embeddings "
          "due to its ability to handle multicollinearity and overfitting better than Logistic Regression.")

# Analyze Neuron Activations
def analyze_neuron_activations(sae_model, embeddings_dict, device):
    activation_dict = {}
    for key, embeddings in embeddings_dict.items():
        if len(embeddings) > 0:
            embeddings_tensor = torch.tensor(embeddings, dtype=torch.float32).to(device)
            activations = sae_model.encode(embeddings_tensor).detach().cpu().numpy()
            activation_dict[key] = activations
        else:
            activation_dict[key] = np.array([])
    return activation_dict

In [4]:

# Main Execution
def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # Data Preparation (assuming final_merged_df is available)
    split_date = pd.Timestamp('2017-01-01')
    train_df = final_merged_df[final_merged_df['date'] < split_date].reset_index(drop=True)
    test_df = final_merged_df[final_merged_df['date'] >= split_date].reset_index(drop=True)
    forecast_horizon = 20
    train_dataset = StockClassificationDataset(train_df, sequence_length=252, forecast_horizon=forecast_horizon)
    test_dataset = StockClassificationDataset(test_df, sequence_length=252, forecast_horizon=forecast_horizon)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
    print("Train samples:", len(train_dataset))
    print("Test samples:", len(test_dataset))

    conditions = [
        {'name': 'withsentiment_general_withmask', 'model_path': '/content/drive/MyDrive/financial_data/model_v2/model_With_Sentiment_With_General_Mask_forecast_20_1_5_20.pth', 'projection_path': '/content/drive/MyDrive/financial_data/model_v2/projection_masked_True_general_True_forecast_20_1_5_20.pth', 'use_sentiment': True, 'use_general': True, 'use_masking': True},
        {'name': 'withsentiment_withmask', 'model_path': '/content/drive/MyDrive/financial_data/model_v2/model_With_Sentiment_No_General_Mask_forecast_20_1_5_20.pth', 'projection_path': '/content/drive/MyDrive/financial_data/model_v2/projection_masked_True_general_False_forecast_20_1_5_20.pth', 'use_sentiment': True, 'use_general': False, 'use_masking': True},
    ]


    for condition in conditions:
        print(f"\n=== Processing Condition: {condition['name']} ===")
        # Set input channels: 15 if both sentiments, 10 if only specific sentiment, 5 if no sentiment
        num_input_channels = 15 if condition['use_sentiment'] and condition['use_general'] else (10 if condition['use_sentiment'] else 5)
        config = PatchTSMixerConfig(context_length=252, patch_length=12, patch_stride=12,
                                    num_input_channels=num_input_channels, num_targets=3, d_model=32, num_layers=8)
        model = PatchTSMixerForTimeSeriesClassification(config)
        model.load_state_dict(torch.load(condition['model_path'], map_location=device))
        model.eval()
        projection = None
        if condition['use_sentiment']:
            projection = nn.Linear(768, 5).to(device)
            projection.load_state_dict(torch.load(condition['projection_path'], map_location=device))
            projection.eval()

        # Extract Hidden States
        train_results = extract_hidden_states(model, train_loader, device, condition['use_sentiment'], condition['use_general'], projection)
        test_results = extract_hidden_states(model, test_loader, device, condition['use_sentiment'], condition['use_general'], projection)

        # Train SAE
        train_all_embeddings = train_results['hidden_states']
        input_dim = train_all_embeddings.shape[1]
        sae_model = TopKSAE(d_model=input_dim, d_sae=8000, k=160)
        sae_model = train_topk_sae(sae_model, train_all_embeddings, epochs=20, device=device)

        # Analyze Activations
        train_embeddings_dict = {}
        test_embeddings_dict = {}
        if condition['use_masking']:
            train_easy_mask = train_results['easy_masks'].astype(bool)
            train_hard_mask = ~train_easy_mask
            train_correct = train_results['predictions'] == train_results['targets']
            train_incorrect = ~train_correct
            train_embeddings_dict[f"{condition['name']}_easy_correct"] = train_results['hidden_states'][np.logical_and(train_easy_mask, train_correct)]
            train_embeddings_dict[f"{condition['name']}_easy_incorrect"] = train_results['hidden_states'][np.logical_and(train_easy_mask, train_incorrect)]
            train_embeddings_dict[f"{condition['name']}_hard_correct"] = train_results['hidden_states'][np.logical_and(train_hard_mask, train_correct)]
            train_embeddings_dict[f"{condition['name']}_hard_incorrect"] = train_results['hidden_states'][np.logical_and(train_hard_mask, train_incorrect)]
            test_easy_mask = test_results['easy_masks'].astype(bool)
            test_hard_mask = ~test_easy_mask
            test_correct = test_results['predictions'] == test_results['targets']
            test_incorrect = ~test_correct
            test_embeddings_dict[f"{condition['name']}_easy_correct"] = test_results['hidden_states'][np.logical_and(test_easy_mask, test_correct)]
            test_embeddings_dict[f"{condition['name']}_easy_incorrect"] = test_results['hidden_states'][np.logical_and(test_easy_mask, test_incorrect)]
            test_embeddings_dict[f"{condition['name']}_hard_correct"] = test_results['hidden_states'][np.logical_and(test_hard_mask, test_correct)]
            test_embeddings_dict[f"{condition['name']}_hard_incorrect"] = test_results['hidden_states'][np.logical_and(test_hard_mask, test_incorrect)]
        else:
            train_correct = train_results['predictions'] == train_results['targets']
            train_embeddings_dict[f"{condition['name']}_correct"] = train_results['hidden_states'][train_correct]
            train_embeddings_dict[f"{condition['name']}_incorrect"] = train_results['hidden_states'][~train_correct]
            test_correct = test_results['predictions'] == test_results['targets']
            test_embeddings_dict[f"{condition['name']}_correct"] = test_results['hidden_states'][test_correct]
            test_embeddings_dict[f"{condition['name']}_incorrect"] = test_results['hidden_states'][~test_correct]
        train_activation_dict = analyze_neuron_activations(sae_model, train_embeddings_dict, device)
        test_activation_dict = analyze_neuron_activations(sae_model, test_embeddings_dict, device)

        # Prepare Labels
        def get_category_label(easy_mask, is_correct):
            return 'Easy + Correct' if easy_mask == 1 and is_correct else 'Easy + Incorrect' if easy_mask == 1 else 'Hard + Correct' if is_correct else 'Hard + Incorrect'

        train_is_correct = train_results['predictions'] == train_results['targets']
        train_labels = [get_category_label(easy, correct) for easy, correct in zip(train_results['easy_masks'], train_is_correct)]
        test_is_correct = test_results['predictions'] == test_results['targets']
        test_labels = [get_category_label(easy, correct) for easy, correct in zip(test_results['easy_masks'], test_is_correct)]

        # Combine Embeddings and Labels for Clustering/Classification
        if condition['use_masking']:
            train_embeddings = np.vstack([train_activation_dict[key] for key in train_embeddings_dict if len(train_activation_dict[key]) > 0])
            test_embeddings = np.vstack([test_activation_dict[key] for key in test_embeddings_dict if len(test_activation_dict[key]) > 0])
            train_labels_array = np.array([label for i, (easy, correct) in enumerate(zip(train_results['easy_masks'], train_is_correct))
                                         for label in [get_category_label(easy, correct)]
                                         if np.any(np.logical_and(train_results['easy_masks'] == easy, train_is_correct == correct))])
            test_labels_array = np.array([label for i, (easy, correct) in enumerate(zip(test_results['easy_masks'], test_is_correct))
                                        for label in [get_category_label(easy, correct)]
                                        if np.any(np.logical_and(test_results['easy_masks'] == easy, test_is_correct == correct))])
        else:
            train_embeddings = np.vstack([train_activation_dict[key] for key in train_embeddings_dict if len(train_activation_dict[key]) > 0])
            test_embeddings = np.vstack([test_activation_dict[key] for key in test_embeddings_dict if len(test_activation_dict[key]) > 0])
            train_labels_array = np.array([label for label, emb in zip(train_labels, train_activation_dict.values()) if len(emb) > 0 for _ in range(len(emb))])
            test_labels_array = np.array([label for label, emb in zip(test_labels, test_activation_dict.values()) if len(emb) > 0 for _ in range(len(emb))])

        # Visualization
        visualize_tsne(train_embeddings, train_labels_array, f"t-SNE of Train Embeddings ({condition['name']})", f"/content/drive/MyDrive/financial_data/result_v2/tsne_train_{condition['name']}.png")
        visualize_tsne(test_embeddings, test_labels_array, f"t-SNE of Test Embeddings ({condition['name']})", f"/content/drive/MyDrive/financial_data/result_v2/tsne_test_{condition['name']}.png")
        visualize_top_neurons(train_activation_dict, train_embeddings_dict.keys(), top_n=10, filename_prefix=f"train_{condition['name']}")

        # Clustering
        print(f"\nClustering for {condition['name']}:")
        perform_clustering(train_embeddings)

        # Classification
        print(f"\nClassification for {condition['name']}:")
        perform_classification(train_embeddings, train_labels_array, test_embeddings, test_labels_array)


if __name__ == "__main__":
    # Assuming final_merged_df is defined
    main()

Using device: cuda
Loaded embedding caches from /content/drive/MyDrive/financial_data/embedding_cache/specific_embeddings_train.npz and /content/drive/MyDrive/financial_data/embedding_cache/general_embeddings_train.npz


Processing Dates: 100%|██████████| 1760/1760 [01:01<00:00, 28.62it/s]


Loaded embedding caches from /content/drive/MyDrive/financial_data/embedding_cache/specific_embeddings_train.npz and /content/drive/MyDrive/financial_data/embedding_cache/general_embeddings_train.npz


Processing Dates: 100%|██████████| 754/754 [00:20<00:00, 36.61it/s]


Train samples: 67284
Test samples: 22701

=== Processing Condition: withsentiment_general_withmask ===


...

SAE Epoch 20/20: 100%|██████████| 1052/1052 [00:46<00:00, 22.65it/s]



Clustering for withsentiment_withmask:
Silhouette Score for 4 clusters: 0.0608

Classification for withsentiment_withmask:

Evaluating Logistic Regression:
Logistic Regression Classification Report:
                  precision    recall  f1-score   support

  Easy + Correct       0.00      0.00      0.00      1582
Easy + Incorrect       0.00      0.00      0.00      1654
  Hard + Correct       0.47      0.98      0.63     10609
Hard + Incorrect       0.45      0.02      0.05      8856

        accuracy                           0.47     22701
       macro avg       0.23      0.25      0.17     22701
    weighted avg       0.39      0.47      0.31     22701


Evaluating Random Forest:


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Random Forest Classification Report:
                  precision    recall  f1-score   support

  Easy + Correct       0.00      0.00      0.00      1582
Easy + Incorrect       0.00      0.00      0.00      1654
  Hard + Correct       0.47      0.97      0.63     10609
Hard + Incorrect       0.40      0.03      0.06      8856

        accuracy                           0.47     22701
       macro avg       0.22      0.25      0.17     22701
    weighted avg       0.38      0.47      0.32     22701


Recommendation: Random Forest is preferred for high-dimensional SAE embeddings due to its ability to handle multicollinearity and overfitting better than Logistic Regression.


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
